In [ ]:
from textprep.vectorizer import *
from textprep.embed import EmbeddingsHandler
import pandas as pd

from utils.download import *
from pulearn.pubase import *


download_from_gdrive("deceptive-opinion.csv")

documents = pd.read_csv("../data/deceptive-opinion.csv")
documents["deceptive"] = documents["deceptive"].apply(lambda x: -1 if x == "truthful" else 1)
documents["label"] = documents["deceptive"].copy()
documents = documents.drop(columns=["hotel", "source", "polarity", "deceptive"])

documents["sequences"], vectorizer = SequenceVectorizer.from_dataframe (
    documents, 
    "text", 
    "label", 
    prep_funcs={
        to_lower: {},
        to_remove_symbols: {}
    }
)

emb_handler = EmbeddingsHandler(vectorizer.text_vocab, pretrained="fasttext-wiki")
embeddings = emb_handler.load_embeddings()

Define a datamodule and an estimator for the classification task

In [ ]:
from pulearn.neural_nets.estimators import CNNEstimator, PUNet
from pulearn.neural_nets.data_modules import DeceptiveOpinionsDataModule

import pytorch_lightning as pl


datamodule = DeceptiveOpinionsDataModule(documents=documents, batch_size=512)

estimator = CNNEstimator(num_classes=1, pretrained_embedding=embeddings)

pu_net = PUNet(
    estimator=estimator, 
    learning_rate=0.01
)

Define a trainer with the available hardware and logging configurations

In [ ]:
# trainer = pl.Trainer (
#     max_epochs=50,
#     accelerator="gpu",
#     devices=1
# )

Find optimal learning rate

In [ ]:
# lr_finder = trainer.tuner.lr_find(pu_net, datamodule, max_lr=1e-2)

Plot of learning rate trials

In [ ]:
# fig = lr_finder.plot(suggest=True)
# fig.show()

Store new learning rate in our classifierm

In [ ]:
# pu_net.learning_rate = lr_finder.suggestion()

Create a checkpointing mechanism to avoid overfitting and a tensorboard logger to visualize results

In [ ]:
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger


model_ckpt = ModelCheckpoint (
    filename="{epoch}-{step}-{val_loss:.2f}",
    save_last=True,
    save_top_k=3,
    monitor="val_loss_epoch",
    mode="min"
)

logger = TensorBoardLogger(save_dir="..\lightning_logs", name="cnn-experiments")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=../lightning_logs/lightning_logs/version_0/

Create a new trainer which will be used for the actual training of the model

In [ ]:
trainer = pl.Trainer (
    default_root_dir="../lightning_logs/lightning_logs/",
    accelerator="gpu",
    devices=1,
    max_steps=50,
    precision=16,
    val_check_interval=3,
    log_every_n_steps=3,
    callbacks=[
        model_ckpt
    ],
    logger=logger,
    fast_dev_run=True
)

Fit model on the data

In [ ]:
trainer.fit(pu_net, datamodule)

Perform a prediction on the test set

In [ ]:
trainer.test(datamodule=datamodule, ckpt_path="best")